[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/04_datenqualitaet.ipynb)

# Sitzung 4 — Von Rohdaten zu erster Struktur

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Letzte Woche hattet ihr eine schöne Sentiment-Zahl. Heute die unbequeme Frage: **war die zugrunde liegende Daten überhaupt gut?** Echte Daten sind messy — und der größte Teil von „AI-Arbeit“ ist Datenaufräumen.

> 💡 „Garbage in, garbage out“: Kein Modell ist besser als seine Daten.

## 0. Setup & Daten erzeugen

In [ ]:
import random, re
from collections import Counter
from datetime import date, timedelta
print('Fertig.')

In [ ]:
"""
Synthetic review corpus generator for the BDA course.

Fictional product: "Nimbus Q2" wireless earbuds, made-up brand "Nimbus Audio".
Composes reviews from varied templates + aspect fragments so most reviews are
UNIQUE (realistic), while deliberately seeding the messiness the course teaches:
  - mixed sentiment, sarcasm, fakes, a few English, junk rows
  - a small number of INTENTIONAL exact duplicates (for the dedup lesson)
  - ground-truth flags (true_sentiment, is_sarcastic, is_fake) for later eval
"""
import random
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"
BRAND = "Nimbus Audio"

POS = ["der Klang ist hervorragend", "satte Bässe", "die Geräuschunterdrückung ist top",
       "der Akku hält den ganzen Tag", "sitzt super bequem", "Bluetooth verbindet sofort",
       "top verarbeitet", "klasse für den Preis", "die App ist übersichtlich",
       "die Passform ist perfekt", "der Sound ist klar und ausgewogen"]
NEG = ["die App stürzt ständig ab", "der rechte Ohrhörer lädt nicht mehr",
       "die Geräuschunterdrückung rauscht", "viel zu teuer", "die Touch-Steuerung reagiert kaum",
       "das Case wirkt billig", "der Akku ist nach einer Stunde leer",
       "die Verbindung bricht ab", "sie fallen leicht aus dem Ohr", "der Bass ist matschig"]

POS_OPENERS = ["Bin begeistert:", "Wirklich gut:", "Kann ich empfehlen –", "Top Kauf.",
               "Sehr zufrieden:", "Absolute Kaufempfehlung.", "Ich liebe sie:",
               "Klare Sache:", "Rundum gelungen:", "Volle Punktzahl:", "Endlich zufrieden:",
               "Was soll ich sagen –", "Genau richtig:", "Bestellung hat sich gelohnt:"]
NEG_OPENERS = ["Enttäuschend:", "Leider schlecht:", "Finger weg –", "Bin frustriert:",
               "Nicht zu empfehlen.", "Schade um das Geld:", "Ärgerlich:",
               "Reklamiert:", "Bin raus:", "Nie wieder:", "Herbe Enttäuschung:",
               "Das war nichts:", "Zurückgeschickt:", "Vorsicht:"]
NEU_TEMPLATES = ["Ganz okay, {a}, aber nichts Besonderes.",
                 "Erfüllt seinen Zweck. {a_cap}.",
                 "Durchschnittlich. {a_cap}, mehr nicht.",
                 "Habe sie seit Kurzem, {a} – kann noch nicht viel sagen."]
SARCASTIC = ["Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
             "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
             "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
             "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
             "Klasse, nach einer Woche nur noch Rauschen. Wirklich durchdacht.",
             "Perfekt, der linke fällt ständig raus. Genau mein Wunsch.",
             "Herrlich, die Verbindung bricht alle fünf Minuten ab. Danke auch.",
             "Sensationell, das Case bricht beim ersten Öffnen. Qualität eben.",
             "Bravo, nach dem Update ist die Hälfte der Funktionen weg.",
             "Fantastisch leise – weil nach zwei Tagen einfach tot."]
FAKE = ["BESTES PRODUKT EVER!!! Kauft bei www.super-deals-guenstig.example!!!",
        "5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
        "Gutschein Code NIMBUS100 auf meiner Seite jetzt klicken!!!",
        "amazing product best quality buy now discount link in profile",
        "TOP TOP TOP unbedingt kaufen billigster preis hier klicken",
        "gratis versand nur heute!!! rabattcode DEAL22 einlösen!!!",
        "beste kopfhoerer der welt jetzt zuschlagen link im profil",
        "WOW einfach WOW kaufen kaufen kaufen bester preis garantiert",
        "unglaublich guenstig hier klicken und sparen sparen sparen",
        "mega angebot heute -70% nur ueber meinen link!!!"]
ENGLISH = [("Sound quality is great but the app is a disaster.", "mixed"),
           ("Battery life is amazing, best earbuds I have owned.", "positive"),
           ("Stopped working after a week, very disappointed.", "negative"),
           ("Comfortable fit and clear sound, happy with the purchase.", "positive"),
           ("The noise cancelling is weak and the case feels cheap.", "negative")]
JUNK = ["", "   ", ".", "???", "kein kommentar", "-", "n/a", "...", "!!", "??", "keine angabe", "test"]

def _pos(rng):
    o = rng.choice(POS_OPENERS); a = rng.sample(POS, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "positive"

def _neg(rng):
    o = rng.choice(NEG_OPENERS); a = rng.sample(NEG, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "negative"

def _mixed(rng):
    p = rng.choice(POS); n = rng.choice(NEG)
    conn = rng.choice([" - aber ", ", allerdings ", ". Leider ", ", jedoch "])
    return f"{p[0].upper()+p[1:]}{conn}{n}.", "mixed"

def _neutral(rng):
    a = rng.choice(POS + NEG)
    t = rng.choice(NEU_TEMPLATES).format(a=a, a_cap=a[0].upper()+a[1:])
    return t, "neutral"

def _rating_for(truth, rng):
    return rng.choice({"positive":[4,5,5],"negative":[1,1,2],"mixed":[2,3,4],
                       "neutral":[3,3,4],"fake":[5,5]}.get(truth,[1,3,5]))

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:   text, truth = _pos(rng)
        elif r < 0.60: text, truth = _neg(rng)
        elif r < 0.72: text, truth = _mixed(rng)
        elif r < 0.80: text, truth = _neutral(rng)
        elif r < 0.88: text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.93: text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98: text, truth = rng.choice(ENGLISH)
        else:          text, truth = rng.choice(JUNK), "junk"
        day = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day)).isoformat(),
            "product": PRODUCT,
            "rating": _rating_for(truth, rng),
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": text in SARCASTIC,
            "is_fake": text in FAKE,
        })
    # seed exactly 3 intentional exact-duplicates for the dedup lesson
    if n > 20:
        for j, src in enumerate([5, 12, 30]):
            rows.append(dict(rows[src], review_id=f"R{n+j:04d}"))
    rng.shuffle(rows)
    return rows

In [ ]:
reviews = generate(300)
print(f'{len(reviews)} Zeilen erzeugt.')

## 1. Erstmal hinschauen

Bevor man rechnet, schaut man in die Daten. Die ersten Zeilen sehen ordentlich aus — aber blättert man weiter, findet man Überraschungen. Führt die Zelle aus und lest ein paar Zeilen bewusst:

In [ ]:
for r in reviews[:15]:
    print(f"[{r['rating']}*] {repr(r['text'][:55])}")

> ✏️ **Fällt euch etwas auf?** Leere Texte, „???“, komische Groß-/Kleinschreibung, englische Sätze? Genau das ist echtes Daten-Chaos.

## 2. Probleme systematisch finden

Auge reicht nicht bei hunderten Zeilen. Wir suchen die Probleme **im Code**.

**a) Leere / Müll-Zeilen** — Texte, die praktisch nichts enthalten:

In [ ]:
muell = [r for r in reviews if len(r['text'].strip()) < 5]
print(f'{len(muell)} Müll-/Leerzeilen')
for r in muell[:5]: print('  ', repr(r['text']))

**b) Duplikate** — dieselbe Bewertung mehrfach. In echten Daten oft **Spam** (copy-paste) oder wiederholte Junk-Einträge — genau das, was man rauswerfen will:

In [ ]:
texte = [r['text'] for r in reviews]
dupe_texte = [t for t, c in Counter(texte).items() if c > 1 and t.strip()]
print(f'{len(dupe_texte)} Texte kommen mehrfach vor')
for t in dupe_texte[:3]: print('  ', repr(t[:55]))

**c) Fremdsprache** — ein paar englische Bewertungen. Eine simple Heuristik: typische englische Wörter suchen. *(Nicht perfekt — echte Sprach-Erkennung ist ein eigenes Thema.)*

In [ ]:
en_woerter = ['the', 'is', 'and', 'my', 'was', 'best', 'quality', 'product']
def wirkt_englisch(t):
    woerter = re.findall(r'[a-zA-Z]+', t.lower())
    treffer = sum(w in en_woerter for w in woerter)
    return treffer >= 2
englisch = [r for r in reviews if wirkt_englisch(r['text'])]
print(f'{len(englisch)} wirken englisch')
for r in englisch[:4]: print('  ', repr(r['text'][:55]))

## 3. Aufräumen

Jetzt bauen wir einen **sauberen** Datensatz. Entscheidungen, die man bewusst trifft (es gibt kein „richtig“ — nur begründet):

- Leere/Müll-Zeilen → **raus** (kein Informationsgehalt)
- Duplikate → **einmal behalten** (sonst zählt man doppelt)
- Englische → **markieren** (behalten, aber getrennt auswertbar)
- Text leicht normalisieren (Leerzeichen trimmen)

In [ ]:
sauber = []
gesehen = set()
for r in reviews:
    text = r['text'].strip()
    if len(text) < 5:               # Müll raus
        continue
    if text in gesehen:             # Duplikat überspringen
        continue
    gesehen.add(text)
    r2 = dict(r)
    r2['text'] = text
    r2['sprache'] = 'en' if wirkt_englisch(text) else 'de'
    sauber.append(r2)

print(f'Vorher: {len(reviews)}  ->  Nachher: {len(sauber)}')
print('Sprachen:', Counter(r['sprache'] for r in sauber))

## 4. Wie viel hat das Aufräumen verändert?

Von ~300 auf ~250 Zeilen — wir haben Müll und Duplikate entfernt, aber die **echten** Bewertungen behalten. Genau so soll Aufräumen sein: das Rauschen raus, das Signal bleibt. Vergleicht eine Kennzahl (Durchschnitts-Sterne) vorher/nachher.

In [ ]:
def schnitt(rows):
    sterne = [r['rating'] for r in rows]
    return round(sum(sterne)/len(sterne), 2)
print('Durchschnitts-Sterne roh   :', schnitt(reviews))
print('Durchschnitts-Sterne sauber:', schnitt(sauber))

> ✏️ **Eure Aufgabe:** Ändert die Aufräum-Regeln oben (z. B. englische auch rauswerfen statt markieren) und schaut, wie sich die Zahlen bewegen. Es gibt keine perfekte Antwort — nur begründete Entscheidungen.

## 5. Auflösung: die eingebauten „Fallen“

Jetzt der Reveal. Der Datensatz wurde absichtlich mit Fallen gebaut — und trägt versteckte **Wahrheits-Marker** (nur für den Kurs, in echten Daten hat man die nie). Vergleicht, wie gut ihr die Probleme gefunden habt:

In [ ]:
fakes = [r for r in reviews if r['is_fake']]
sarkasmus = [r for r in reviews if r['is_sarcastic']]
print(f'Eingebaute Fake-Reviews : {len(fakes)}')
print(f'Eingebauter Sarkasmus   : {len(sarkasmus)}')
print()
print('Beispiel Fake   :', repr(fakes[0]['text'][:55]) if fakes else '-')
print('Beispiel Sarkasmus:', repr(sarkasmus[0]['text'][:55]) if sarkasmus else '-')

> 💡 **Kernpunkt:** Manche Probleme (leer, Duplikat) findet man mechanisch. Andere — **Fakes, Sarkasmus** — sieht man den Daten *nicht* einfach an. Genau die werden uns noch beschäftigen: sie verzerren jede Analyse, und ein LLM fällt oft darauf herein. *(Sitzung 9 & 10.)*

## 6. Geschafft — und Ausblick

Ihr habt aus rohen, messy Daten einen sauberen, strukturierten Datensatz gemacht — die unspektakuläre, aber entscheidende Grundlage jeder Analyse.

**Nächste Woche (04.11):** Schluss mit reinem Sentiment. Wir extrahieren **Themen** — worüber reden die Leute eigentlich?

> 💡 „Most of AI is data wrangling.“ Heute habt ihr gesehen, warum.